In [ ]:
# 1. INSTALAÇÃO DAS DEPENDÊNCIAS
# Adicionada a biblioteca statsmodels para suportar o ARIMA (caso já não esteja instalada no Colab)
!pip install flask flask-cors firebase-admin tensorflow pandas numpy joblib scikit-learn pyngrok statsmodels

import os
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from flask import Flask, jsonify, request
from flask_cors import CORS
import firebase_admin
from firebase_admin import credentials, db
from google.colab import files
from pyngrok import ngrok
# === NOVO: ADIÇÃO DO ARIMA ===
from statsmodels.tsa.arima.model import ARIMAResults

# 2. CARREGAR ARQUIVOS NECESSÁRIOS (Adicionado o ficheiro do ARIMA)
print("Por favor, faz o upload dos ficheiros necessários (.json, .keras, .pkl, .pkl do ARIMA):")
uploaded = files.upload()

# 3. CONFIGURAÇÃO E AUTENTICAÇÃO DO NGROK
ngrok.set_auth_token("3E6FIuJGibkJKDB6xQoe9mEXqRW_6sfmiqdyYkc8xfB36qBnk")

# 4. INICIALIZAÇÃO DO FLASK E CONFIGURAÇÃO DE CORS
app = Flask(__name__)
CORS(app)

# 5. CONEXÃO COM O FIREBASE REALTIME DATABASE
if not firebase_admin._apps:
    cred = credentials.Certificate("serviceAccountKey.json")
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://consumo-conciente-default-rtdb.firebaseio.com/'
    })

# 6. CARREGAMENTO DOS MODELOS DE INTELIGÊNCIA ARTIFICIAL
MODELO_PATH = 'modelo_lstm_energia.keras'
SCALER_PATH = 'scaler_energia.pkl'
# === NOVO: ADIÇÃO DO ARIMA ===
# Certifica-te de que o teu ficheiro ARIMA guardado via joblib/pickle tem este nome exato
ARIMA_PATH = 'modelo_arima_energia.pkl'

print("A carregar o modelo LSTM, Scaler e ARIMA...")
modelo = tf.keras.models.load_model(MODELO_PATH)
scaler = joblib.load(SCALER_PATH)
# === NOVO: ADIÇÃO DO ARIMA ===
modelo_arima = joblib.load(ARIMA_PATH)
print("Modelos carregados com sucesso!")

# --- CONFIGURAÇÕES DO DATASET REAL ---
COLUNAS_MONITORADAS = ['consumo_total_pzem', 'corrente', 'potencia', 'voltagem']
INDICE_ALVO = COLUNAS_MONITORADAS.index('consumo_total_pzem')
JANELA_PADRAO = 60


# 7. FUNÇÕES AUXILIARES (TRATAMENTO DE DADOS E PREVISÃO)
def buscar_dados_pzem(quantidade):
    """Busca o histórico do Firebase e aplana a estrutura hierárquica (Data -> Hora)"""
    ref = db.reference('leituras')
    dados = ref.get()

    if not dados:
        return pd.DataFrame()

    linhas_achatadas = []

    for data in sorted(dados.keys()):
        horas = dados[data]
        if isinstance(horas, dict):
            for hora in sorted(horas.keys()):
                atributos = horas[hora]
                if isinstance(atributos, dict):
                    registro = atributos.copy()

                    if 'consumo' in registro and 'consumo_total_pzem' not in registro:
                        registro['consumo_total_pzem'] = registro['consumo']
                    elif 'consumo_acumulado' in registro and 'consumo_total_pzem' not in registro:
                        registro['consumo_total_pzem'] = registro['consumo_acumulado']

                    linhas_achatadas.append(registro)

    df = pd.DataFrame(linhas_achatadas)

    if df.empty:
        return df

    for col in COLUNAS_MONITORADAS:
        if col not in df.columns:
            df[col] = 0.0

    df_filtrado = df[COLUNAS_MONITORADAS].dropna()
    return df_filtrado.tail(quantidade)


def prever_passos_futuros_hibrido(df, passos_solicitados):
    """Aplica o modelo LSTM e ARIMA para gerar uma previsão combinada (Híbrida)"""

    # --- 1. PREVISÃO ITERATIVA DO LSTM ---
    dados_normalizados = scaler.transform(df[COLUNAS_MONITORADAS])
    janela_atual = dados_normalizados.copy()
    previsoes_lstm = []

    for _ in range(passos_solicitados):
        input_lstm = np.reshape(janela_atual, (1, janela_atual.shape[0], janela_atual.shape[1]))
        predicao_escalonada = modelo.predict(input_lstm, verbose=0)

        linha_fake = np.zeros((1, len(COLUNAS_MONITORADAS)))
        linha_fake[0, INDICE_ALVO] = predicao_escalonada[0, 0]

        valor_real_consumo = scaler.inverse_transform(linha_fake)[0, INDICE_ALVO]
        previsoes_lstm.append(float(valor_real_consumo))

        proxima_linha = janela_atual[-1].copy()
        proxima_linha[INDICE_ALVO] = predicao_escalonada[0, 0]
        janela_atual = np.vstack([janela_atual[1:], proxima_linha])

    # --- 2. PREVISÃO DO ARIMA (=== NOVO ===) ---
    # O ARIMA necessita apenas da série temporal histórica real do alvo (consumo)
    serie_temporal = df['consumo_total_pzem'].values

    try:
        # Atualiza o modelo ARIMA com os dados mais recentes coletados do Firebase
        arima_atualizado = modelo_arima.append(serie_temporal, refit=False)
        # Realiza a previsão para N passos à frente
        previsoes_arima = arima_atualizado.forecast(steps=passos_solicitados)
        previsoes_arima = [float(x) for x in previsoes_arima]
    except Exception as e:
        # Fallback de segurança: Caso o append falhe por incompatibilidade de formato
        # faz o forecast direto a partir do último estado conhecido do modelo carregado
        previsoes_arima = modelo_arima.forecast(steps=passos_solicitados)
        previsoes_arima = [float(x) for x in previsoes_arima]

    # --- 3. COMBINAÇÃO HÍBRIDA (Média Ponderada) ---
    # Podes ajustar os pesos (ex: 0.6 para LSTM e 0.4 para ARIMA) baseado nas métricas de treino
    peso_lstm = 0.5
    peso_arima = 0.5

    previsoes_hibridas = []
    for p_lstm, p_arima in zip(previsoes_lstm, previsoes_arima):
        valor_hibrido = (p_lstm * peso_lstm) + (p_arima * peso_arima)
        previsoes_hibridas.append(valor_hibrido)

    return previsoes_hibridas


# --- 8. ENDPOINTS DA API REST ---

@app.route('/api/status_atual', methods=['GET'])
def obter_status_atual():
    """Retorna os dados em tempo real mais recentes do sensor e a previsão híbrida imediata (+1 passo)"""
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({
            'erro': f'Histórico insuficiente no Firebase. São necessárias {JANELA_PADRAO} amostras, mas o sistema possui apenas {len(df)}.'
        }), 400

    ultima_leitura = df.iloc[-1].to_dict()
    # Mudança para a nova função híbrida
    previsao_imediata = prever_passos_futuros_hibrido(df, passos_solicitados=1)

    return jsonify({
        'status': 'sucesso',
        'leitura_atual': ultima_leitura,
        'previsao_proximo_passo': previsao_imediata[0]
    })


@app.route('/api/previsao_futura', methods=['GET'])
def obter_previsao_futura():
    """Retorna a projeção híbrida de N passos à frente definida pelo parâmetro '?passos=' na URL"""
    passos = int(request.args.get('passos', 5))
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({'erro': 'Histórico insuficiente para realizar previsões futuras.'}), 400

    # Mudança para a nova função híbrida
    projecao = prever_passos_futuros_hibrido(df, passos_solicitados=passos)

    return jsonify({
        'status': 'sucesso',
        'passos_calculados': passos,
        'valores_projetados': projecao
    })


@app.route('/', methods=['GET'])
def testar_servidor():
    return jsonify({'mensagem': 'API de Eficiência Energética Híbrida (LSTM + ARIMA) a rodar com sucesso!'})


# --- 9. EXECUÇÃO DO TUNEL E SERVIDOR ---
if __name__ == '__main__':
    public_url = ngrok.connect(5000)
    print("=" * 60)
    print("URL PÚBLICA DA API (Utiliza esta no teu Front-end):")
    print(public_url)
    print("=" * 60)

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )